In [1]:
import clr  # From pythonnet
import os
import random
import numpy as np
import pandas as pd
from sklearn.cluster import MeanShift
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Paths
dwsim_path = r"C:\Users\user\AppData\Local\DWSIM\\"
sim_path = r"C:\Users\user\Documents\SAF_kinetics_paper_code_with_WGS\SAF_kinetics_paper_code_with_WGS\dwsim_pilot_model_parametric.dwxmz"

In [3]:
# 1. Setup paths to DWSIM installation
clr.AddReference(os.path.join(dwsim_path, "DWSIM.Automation.dll"))
clr.AddReference(os.path.join(dwsim_path, "DWSIM.Interfaces.dll"))
clr.AddReference(os.path.join(dwsim_path, "ThermoCS\\ThermoCS.dll"))

from DWSIM.Automation import Automation3
from System import String

# 2. Initialize the Automation Manager
interf = Automation3()

# 3. Load an existing simulation (.dwxmz)
Flowsheet = interf.LoadFlowsheet(sim_path)

In [4]:
H2 = Flowsheet.GetFlowsheetSimulationObject('H2').GetAsObject()
CO = Flowsheet.GetFlowsheetSimulationObject('CO').GetAsObject()
compressor = Flowsheet.GetFlowsheetSimulationObject('C-1').GetAsObject()
cooler = Flowsheet.GetFlowsheetSimulationObject('CL-1').GetAsObject()
syncrude = Flowsheet.GetFlowsheetSimulationObject('syncrude').GetAsObject()
syncrude_phase = syncrude.GetPhase('Overall')
PFR_1 = Flowsheet.GetFlowsheetSimulationObject('PFR-1').GetAsObject()
E_reactor = Flowsheet.GetFlowsheetSimulationObject('E1').GetAsObject()
PFR_1.set_dV(0.1)

In [5]:
v0 = 730 # L / min
v0 = v0 / ( 1000 * 60 ) # m3 / s

F0 = v0 * 1e5 / ( 8.314 * 273 ) # mol / s

H2_CO_in = 2
F_H2_in = F0 * (H2_CO_in / ( 1 + H2_CO_in )) # mol / s
F_CO_in = F0 - F_H2_in

H2.SetMolarFlow(F_H2_in)
CO.SetMolarFlow(F_CO_in)
compressor.POut = 2e6
cooler.OutletTemperature = 300 + 273
PFR_1.CatalystLoading = 1648

# Ligar o solver
interf.CalculateFlowsheet2(Flowsheet)

# check if solved
if not Flowsheet.Solved:
    interf.SaveFlowsheet(Flowsheet, sim_path, True)
    raise ValueError(f'Something went wrong at index {i}, check your simulation.')

interf.SaveFlowsheet(Flowsheet, sim_path, True)

In [6]:
E_reactor.EnergyFlow

-26.96622157152809

In [7]:
for component in syncrude_phase.Compounds.keys():
    print(component, syncrude_phase.Compounds[component].MolarFlow)

Methane 0.008233774498748717
Ethane 0.005506502618110199
Propane 0.004877620548460938
N-butane 0.0021093611246099943
N-pentane 0.0018518234269142577
N-hexane 0.0015855173137391341
N-heptane 0.0013275674940490749
N-octane 0.0010906648199073457
N-nonane 0.0008821837554958283
N-decane 0.000704770679830653
N-undecane 0.000557655998336561
N-dodecane 0.00043804033219183025
N-tridecane 0.00034220503241235117
N-tetradecane 0.00026625566469166506
N-pentadecane 0.0002065467217104372
N-hexadecane 0.00015987976430955865
N-heptadecane 0.00012356122132383792
N-octadecane 9.538353219916387e-05
N-nonadecane 7.357071065671455e-05
N-heneicosane 4.369802684170559e-05
N-docosane 3.365983649903838e-05
N-tetracosane 1.995946137900205e-05
N-pentacosane 1.536676096750618e-05
N-hexacosane 1.182986952541491e-05
N-heptacosane 9.106508095746199e-06
N-octacosane 7.009797586470591e-06
N-nonacosane 5.395678909731024e-06
N-eicosane 5.671225737474627e-05
Water 0.15337476864778868
Hydrogen 0.012581615957058949
Carbon m